# Giai đoạn 18: Kiểm tra số liệu cho bản thảo v8 (CPU, ~5 phút)

Hai kiểm tra chỉ đọc dữ liệu, không chạy mô hình nào:

| Phần | Nội dung | Dùng cho |
|---|---|---|
| 18A | **Guard gap:** khoảng cách thời gian thực giữa trial calib cuối và trial test đầu, cho từng bệnh nhân và từng mức đánh giá | Mục 2.1, thay câu "about 8 s" |
| 18B | **Log bộ lọc:** thông số thiết kế MNE, đáp ứng xung của từng bước và toàn chuỗi, độ trễ thời gian sau resample, tần số lấy mẫu từng bệnh nhân | Mục 2.1, Supplementary Table S2 |

Chỉ tải các file `events.tsv` và `*.json` nhỏ của ds004752 từ OpenNeuro (vài MB).

In [ ]:
#@title 1. Mount Drive + đường dẫn + cài đặt
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
DATA     = f'{ROOT}/data'
OUT      = f'{ROOT}/runs/checks18'
BIDS     = '/content/ds004752_meta'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
os.makedirs(OUT, exist_ok=True)
!pip -q install mne==1.13.2 awscli
!aws s3 sync --no-sign-request s3://openneuro.org/ds004752 {BIDS} --exclude "*" --include "*_events.tsv" --include "*_eeg.json" --include "*_ieeg.json" --only-show-errors
print('events.tsv:', len(glob.glob(f'{BIDS}/sub-*/ses-*/*/*_events.tsv')), '| sidecars:', len(glob.glob(f'{BIDS}/sub-*/ses-*/*/*.json')))

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/gapcheck.py
"""
Guard gap between calibration and test (read-only check for the manuscript, Section 2.1).

The preprocessing chain (notch -> band-pass -> resampling) has an effective impulse response of at most +-3.36 s
(|h| > 1e-6 of the peak; measured on the executed function). Encoder and lagged features are computed within each
trial / window. Calibration and test are therefore separated if, in recording time,
  > 3.36 s  no filtered test sample depends on a raw calibration sample, and vice versa;
  > 6.72 s  no raw sample influences both a calibration and a test sample.
For every patient this reports the separation between the end of the last calibration trial and the start of the
first test trial at 20 % calibration (the smallest separation: smaller calibration fractions keep the same test set
and end calibration earlier).

  ds004752   exact separation from the BIDS event files (onset, duration of the trials, matched by session and
             nTrial) if --bids is given; otherwise the duration of the skipped trial(s), which is a lower bound when
             trials do not overlap in time. Calibration and test in different sessions = separate recordings.
  ScaleOLDNEW (--scales) exact: windows are consecutive within a block; different blocks = separate files.

Usage
-----
python -m s2i.gapcheck --data DATA_DIR [DATA_DIR ...] [--bids BIDS_DIR] [--scales SCALES_PRIMARY_DIR] --out OUT.md
"""
import argparse
import glob
import os
import sys


import torch

FS = 200
IR = 3.36                                # max effective half-support of the chain over the input rates (filtlog)


def n_cal(n):
    return max(3, int(round(n * 0.2)))


def bids_events(bids, sub):
    import pandas as pd
    out = {}
    for ses_dir in sorted(glob.glob(os.path.join(bids, sub, "ses-*"))):
        ses = os.path.basename(ses_dir)
        f = glob.glob(os.path.join(ses_dir, "ieeg", "*events.tsv")) or glob.glob(os.path.join(ses_dir, "eeg", "*events.tsv"))
        if not f:
            continue
        ev = pd.read_csv(f[0], sep="\t")
        if "nTrial" not in ev:
            continue
        for _, r in ev.iterrows():
            if pd.notna(r["nTrial"]):
                out[(ses, int(r["nTrial"]))] = (float(r["onset"]), float(r["duration"]))
    return out


def ds_rows(data, bids):
    rows = []
    for p in sorted(glob.glob(os.path.join(data, "sub-[0-9][0-9].pt"))):
        sub = os.path.basename(p)[:-3]
        d = torch.load(p, weights_only=False)
        b = d["trial_bounds"].numpy()
        ses = list(d.get("trial_session", ["ses-01"] * len(b)))
        tid = list(d.get("trial_id", [None] * len(b)))
        n = len(b)
        nc = n_cal(n)
        last, first = nc - 1, nc + 1
        if first >= n:
            continue
        skipped = (b[last + 1:first, 1] - b[last + 1:first, 0]).sum() / FS
        same = ses[last] == ses[first]
        exact, how = None, "lower bound (skipped trial duration)"
        if bids and same and tid[last] is not None:
            ev = bids_events(bids, sub)
            a, c = ev.get((ses[last], int(tid[last]))), ev.get((ses[first], int(tid[first])))
            if a and c:
                exact, how = c[0] - (a[0] + a[1]), "exact (BIDS events)"
        if not same:
            sep, how = None, "different sessions (separate recordings)"
        else:
            sep = exact if exact is not None else skipped
        rows.append({"patient": sub, "trials": n, "calib": nc, "same_session": same,
                     "gap_trial_s": skipped, "separation_s": sep, "how": how})
    return rows


def scales_rows(data):
    rows = []
    for p in sorted(glob.glob(os.path.join(data, "sub-P*.pt"))):
        if p.endswith("_normstats.pt"):
            continue
        d = torch.load(p, weights_only=False)
        blk, st = list(d["window_block"]), list(d["window_start"])
        b = d["trial_bounds"].numpy()
        n = len(b)
        nc = n_cal(n)
        last, first = nc - 1, nc + 1
        same = blk[last] == blk[first]
        sep = (st[first] - (st[last] + b[last, 1] - b[last, 0])) / FS if same else None
        rows.append({"patient": os.path.basename(p)[:-3], "trials": n, "calib": nc, "same_session": same,
                     "gap_trial_s": (b[nc, 1] - b[nc, 0]) / FS, "separation_s": sep,
                     "how": "exact (window positions)" if same else "different blocks (separate files)"})
    return rows


def table(title, rows):
    L = [f"## {title}", "", "| patient | trials / windows | calibration | same session / block | skipped trial (s) | "
         "separation (s) | > 3.36 s | > 6.72 s | how |", "|---|---|---|---|---|---|---|---|---|"]
    for r in rows:
        s = r["separation_s"]
        ok1 = "yes" if (s is None or s > IR) else "**NO**"
        ok2 = "yes" if (s is None or s > 2 * IR) else "**NO**"
        L.append(f"| {r['patient']} | {r['trials']} | {r['calib']} | {'yes' if r['same_session'] else 'no'} | "
                 f"{r['gap_trial_s']:.2f} | {'separate recordings' if s is None else f'{s:.2f}'} | {ok1} | {ok2} | {r['how']} |")
    same = [r["separation_s"] for r in rows if r["separation_s"] is not None]
    lb = sum(r["how"].startswith("lower bound") for r in rows)
    if same:
        L += ["", f"- minimum separation within one recording: {min(same):.2f} s ({len(same)} patients"
              + (f"; {lb} of them lower bounds from the skipped trial" if lb else "") + "); "
              f"{len(rows) - len(same)} patients have calibration and test in different recordings"]
    else:
        L += ["", "- calibration and test lie in different recordings for every patient"]
    return L + [""]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", nargs="*", default=[], help="ds004752 data folders (e.g. s2i_v1_noqc s2i_v1 s2i_qc_calib)")
    ap.add_argument("--bids", default=None)
    ap.add_argument("--scales", default=None, help="ScaleOLDNEW primary data folder (sub-P*.pt)")
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    L = ["# Guard gap between calibration and test (20 % calibration)", "",
         f"Effective impulse response of the preprocessing chain: +-{IR} s (|h| > 1e-6 of peak).", ""]
    for dd in a.data:
        L += table(f"ds004752: {os.path.basename(dd.rstrip('/'))}", ds_rows(dd, a.bids if a.bids and os.path.isdir(a.bids) else None))
    if a.scales:
        L += table("ScaleOLDNEW: " + os.path.basename(a.scales.rstrip("/")), scales_rows(a.scales))
    out = "\n".join(L)
    print(out)
    open(a.out, "w", encoding="utf-8").write(out + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/filtlog.py
"""
Filter log for the manuscript (Section 2.1, Supplementary Table S2): the MNE design messages of the exact calls made by
s2i.data.preprocess_ds004752.filt_resample, and the measured impulse responses of each step and of the whole chain.

For each input rate (4096 Hz = ScaleOLDNEW, 4000 / 2000 Hz = ds004752 SEEG, 200 Hz = ds004752 scalp EEG of patients
1-9) a unit impulse in the middle of a 60-s signal is passed through
  notch      Raw.notch_filter(NOTCH < Nyquist - 1)      (as in filt_resample)
  band-pass  Raw.filter(0.5, min(70, Nyquist - 1))
  chain      filt_resample itself (notch -> band-pass -> resampling to 200 Hz)
and the support where |h| > 1e-6 of the peak, and the energy beyond +-3.3 s, are reported.

Usage
-----
python -m s2i.filtlog --out OUT.md
"""
import argparse
import logging
import sys

import numpy as np

RATES = (4096.0, 4000.0, 2000.0, 200.0)
THR = 1e-6


class _Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.lines = []

    def emit(self, rec):
        self.lines.append(rec.getMessage())


def extent(h, fs):
    a = np.abs(h) / np.abs(h).max()
    c = int(np.argmax(np.abs(h)))
    idx = np.where(a > THR)[0]
    t = (np.arange(len(h)) - c) / fs
    e = h ** 2 / (h ** 2).sum()
    sym = float(np.max(np.abs(h[c - 50:c + 51] - h[c - 50:c + 51][::-1])) / np.abs(h).max())
    return t[idx.min()], t[idx.max()], float(e[np.abs(t) > 3.3].sum()), sym


def sidecar_rates(bids):
    """{subject: (scalp EEG Hz, SEEG Hz)} from *_eeg.json / *_ieeg.json (SamplingFrequency); sets of rates if mixed."""
    import glob
    import json
    import os
    out = {}
    for sd in sorted(glob.glob(os.path.join(bids, "sub-*"))):
        rates = {}
        for mod in ("eeg", "ieeg"):
            fs = set()
            for f in glob.glob(os.path.join(sd, "ses-*", mod, f"*_{mod}.json")):
                v = json.load(open(f)).get("SamplingFrequency")
                if v is not None:
                    fs.add(float(v))
            rates[mod] = fs.pop() if len(fs) == 1 else (None if not fs else sorted(fs))
        out[os.path.basename(sd)] = (rates["eeg"] if not isinstance(rates["eeg"], list) else None,
                                     rates["ieeg"] if not isinstance(rates["ieeg"], list) else None)
        if any(isinstance(v, list) for v in rates.values()):
            print(f"{os.path.basename(sd)}: several rates across sessions {rates}")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--bids", default=None, help="BIDS root with the *_eeg.json / *_ieeg.json sidecars (optional)")
    ap.add_argument("--extra_rates", type=float, nargs="*", default=[],
                    help="further input rates to measure (rates found in the sidecars are added automatically)")
    a = ap.parse_args()
    if a.bids:
        a.extra_rates = list(a.extra_rates) + [r for fe_fi in sidecar_rates(a.bids).values() for r in fe_fi if r]
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    import mne
    from s2i.data.preprocess_ds004752 import BAND, FS, NOTCH, filt_resample
    log = logging.getLogger("mne")
    L = [f"# Filter log (MNE {mne.__version__}; the calls of filt_resample)", "",
         f"Support = where |h| > {THR:g} of the peak; energy = share of h² beyond ±3.3 s; asymmetry = max |h(t) − h(−t)| "
         f"near the peak / peak.", "",
         "| input rate | step | output rate | support (s) | energy beyond ±3.3 s | asymmetry |", "|---|---|---|---|---|---|"]
    design = []
    for sf in RATES:
        n = int(sf * 60)
        x = np.zeros((1, n))
        x[0, n // 2] = 1.0
        nyq = sf / 2
        notch = [f for f in NOTCH if f < nyq - 1]
        steps = []
        for name in ("notch", "band-pass", "chain"):
            raw = mne.io.RawArray(x.copy(), mne.create_info(["imp"], sf, "eeg"), verbose=False)
            h = _Collect()
            log.addHandler(h)
            old = log.level
            log.setLevel(logging.INFO)
            try:
                if name == "notch":
                    if not notch:
                        steps.append((name, None, sf, ["not applied (100 Hz and 50 Hz handled as in filt_resample)"]))
                        continue
                    y = raw.notch_filter(notch, picks="all", verbose=True).get_data()[0]
                    fs_out = sf
                elif name == "band-pass":
                    y = raw.filter(BAND[0], min(BAND[1], nyq - 1.0), picks="all", verbose=True).get_data()[0]
                    fs_out = sf
                else:
                    y = filt_resample(raw)[0]
                    fs_out = FS
            finally:
                log.removeHandler(h)
                log.setLevel(old)
            steps.append((name, y, fs_out, h.lines))
        for name, y, fs_out, lines in steps:
            if y is None:
                L.append(f"| {sf:g} Hz | {name} | – | not applied (notch {list(NOTCH)} Hz ≥ Nyquist − 1) | – | – |")
                continue
            lo, hi, e, sym = extent(y, fs_out)
            L.append(f"| {sf:g} Hz | {name} | {fs_out:g} Hz | {lo:+.2f} / {hi:+.2f} | {e:.1e} | {sym:.1e} |")
            design.append((sf, name, [ln for ln in lines if ln.strip()]))
    # ---- timing: delay of a 10 Hz sinusoid through the whole chain, per input rate
    delays = {}
    for sf in RATES + tuple(sorted(set(a.extra_rates) - set(RATES))):
        n = int(sf * 60)
        t = np.arange(n) / sf
        z = filt_resample(mne.io.RawArray(np.sin(2 * np.pi * 10 * t)[None], mne.create_info(["s"], sf, "eeg"),
                                          verbose=False))[0]
        to = np.arange(len(z)) / FS
        mid = slice(len(z) // 4, 3 * len(z) // 4)
        ph = np.arctan2(z[mid] @ np.cos(2 * np.pi * 10 * to[mid]), z[mid] @ np.sin(2 * np.pi * 10 * to[mid]))
        delays[sf] = float(-1e3 * ph / (2 * np.pi * 10))
    L += ["", "## Timing of the chain output for a 60-s test signal only (10 Hz sinusoid; negative = output leads the input)",
          "", "The shift of MNE's FFT resampling depends on the exact record length; these values hold for 60-s "
          "signals only. Per-recording values: `python -m s2i.resample_check`.", "",
          "| input rate | delay (ms) |", "|---|---|"]
    L += [f"| {sf:g} Hz | {d:+.3f} |" for sf, d in delays.items()]
    if a.bids:
        L += ["", "## Per patient: sampling rates (BIDS sidecars); offsets below assume 60-s records "
              "(NOT the real recordings; see s2i.resample_check)", "",
              "| patient | scalp EEG Hz | SEEG Hz | scalp delay (ms) | SEEG delay (ms) | SEEG − scalp (ms) |",
              "|---|---|---|---|---|---|"]
        for sub, (fe, fi) in sidecar_rates(a.bids).items():
            de = delays.get(fe) if fe else None
            di = delays.get(fi) if fi else None
            off = f"{di - de:+.3f}" if de is not None and di is not None else "rate not measured"
            L.append(f"| {sub} | {fe or '?'} | {fi or '?'} | {'' if de is None else f'{de:+.3f}'} | "
                     f"{'' if di is None else f'{di:+.3f}'} | {off} |")
    L += ["", "## MNE design messages", ""]
    for sf, name, lines in design:
        L.append(f"**{sf:g} Hz, {name}**")
        L += ["```"] + lines + ["```", ""]
    out = "\n".join(L)
    print(out)
    open(a.out, "w", encoding="utf-8").write(out + "\n")


if __name__ == "__main__":
    main()

## 18A. Guard gap

Ba thư mục ds004752 ứng với ba mức đánh giá, thêm ScaleOLDNEW (mức chính).

Cột "how" cho biết khoảng cách được tính thế nào:
- **exact:** tính chính xác từ onset và duration trong event file;
- **lower bound:** chỉ biết cận dưới, là độ dài trial bị bỏ qua.

Hai ngưỡng: > 3,36 s nghĩa là không mẫu test nào phụ thuộc mẫu raw của calib; > 6,72 s nghĩa là không mẫu raw nào ảnh hưởng đồng thời calib và test.

In [ ]:
#@title 18A
DIRS = ' '.join(f'{DATA}/{d}' for d in ('s2i_v1_noqc', 's2i_v1', 's2i_qc_calib') if os.path.isdir(f'{DATA}/{d}'))
SC = f'{DATA}/scales_s2i/primary'
SC_ARG = f'--scales {SC}' if os.path.isdir(SC) else ''
!python -m s2i.gapcheck --data {DIRS} --bids {BIDS} {SC_ARG} --out {OUT}/gapcheck.md

## 18B. Log bộ lọc, đáp ứng xung và độ trễ sau resample

In [ ]:
#@title 18B
!python -m s2i.filtlog --bids {BIDS} --out {OUT}/filtlog.md 2>/dev/null
from IPython.display import Markdown, display
txt = open(f'{OUT}/filtlog.md', encoding='utf-8').read()
display(Markdown(txt.split('## MNE design messages')[0]))
print('(Thông số thiết kế MNE đầy đủ nằm trong', f'{OUT}/filtlog.md)')

## Gửi lại cho mình
- **Toàn bộ output 18A**, nhất là dòng "minimum separation" của từng thư mục.
- **Phần bảng của 18B:** support, độ trễ theo tần số lấy mẫu và bảng theo bệnh nhân.

Các số này sẽ thay các chỗ `[CHECK 18A]` và `[CHECK 18B]` trong bản thảo v8. Chưa có output thì không điền số nào.